In [ ]:
# Cell Q1: load Qwen3-8B on the T4 GPU (4-bit)
!pip install -q transformers accelerate bitsandbytes

import torch, json, re, time
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

QWEN_ID = "Qwen/Qwen3-8B"
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(QWEN_ID)
model = AutoModelForCausalLM.from_pretrained(
    QWEN_ID, quantization_config=bnb, device_map="auto",
)
print("Qwen3-8B loaded on:", next(model.parameters()).device)

def qwen_generate(prompt, max_new_tokens=256):
    # Disable Qwen3 "thinking" mode to keep output short and parseable
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        do_sample=True, temperature=0.3, top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

In [ ]:
# Cell Q2: Qwen promise judgment, 3 runs per concept, saved incrementally
CONCEPTS_IN = Path("/content/drive/MyDrive/tod/exchange/described.jsonl")
JUDGE_OUT   = Path("/content/drive/MyDrive/tod/exchange/judged_qwen.jsonl")
# If you dragged the file into Colab instead, use these two lines instead:
# CONCEPTS_IN = Path("/content/described.jsonl")
# JUDGE_OUT   = Path("/content/judged_qwen.jsonl")

N_RUNS = 3

JUDGE_PROMPT = """Rate the 'promise' of the emerging technology concept below on a
scale of 1 to 9, judging novelty, potential technical impact, and problem-solving
potential. Score unfounded confidence low.

[Concept] {concept}

Output JSON only, no other text: {{"score": <1-9>, "reason": "<two sentences max>"}}
"""

def parse_score(text):
    m = re.search(r'"score"\s*:\s*([1-9])', text)
    return int(m.group(1)) if m else None

concepts = [json.loads(l) for l in open(CONCEPTS_IN)]
print(f"Judging {len(concepts)} concepts, {N_RUNS} runs each")

with open(JUDGE_OUT, "w") as f:
    for c in concepts:
        prompt = JUDGE_PROMPT.format(concept=c["description"])
        scores = []
        for r in range(N_RUNS):
            raw = qwen_generate(prompt)
            s = parse_score(raw)
            scores.append(s)
            print(f"  {c['concept_id']} run {r+1}: score={s}")
        valid = [s for s in scores if s is not None]
        median = sorted(valid)[len(valid) // 2] if valid else None
        rec = {"concept_id": c["concept_id"], "judge": "qwen3-8b",
               "scores": scores, "median_score": median}
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        f.flush()  # persist after each concept in case the session drops
        print(f"  -> {c['concept_id']} median={median}")

print("Saved:", JUDGE_OUT)

In [ ]:
# Cell Q3: Qwen judgment on the leakage-mitigated (blind) descriptions
BLIND_IN  = Path("/content/drive/MyDrive/tod/exchange/described_blind.jsonl")
BLIND_OUT = Path("/content/drive/MyDrive/tod/exchange/judged_qwen_blind.jsonl")
# If you use direct upload instead of Drive:
# BLIND_IN  = Path("/content/described_blind.jsonl")
# BLIND_OUT = Path("/content/judged_qwen_blind.jsonl")

N_RUNS = 3

JUDGE_PROMPT = """Rate the 'promise' of the emerging technology concept below on a
scale of 1 to 9, judging novelty, potential technical impact, and problem-solving
potential. Score unfounded confidence low.

[Concept] {concept}

Output JSON only, no other text: {{"score": <1-9>, "reason": "<two sentences max>"}}
"""

def parse_score(text):
    m = re.search(r'"score"\s*:\s*([1-9])', text)
    return int(m.group(1)) if m else None

concepts = [json.loads(l) for l in open(BLIND_IN)]
print(f"Judging {len(concepts)} blind concepts, {N_RUNS} runs each")

with open(BLIND_OUT, "w") as f:
    for c in concepts:
        prompt = JUDGE_PROMPT.format(concept=c["description_blind"])
        scores = []
        for r in range(N_RUNS):
            raw = qwen_generate(prompt)
            s = parse_score(raw)
            scores.append(s)
            print(f"  {c['concept_id']} run {r+1}: score={s}")
        valid = [s for s in scores if s is not None]
        median = sorted(valid)[len(valid)//2] if valid else None
        rec = {"concept_id": c["concept_id"], "judge": "qwen3-8b",
               "condition": "blind", "scores": scores, "median_score": median}
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        f.flush()
        print(f"  -> {c['concept_id']} median={median}")

print("Saved:", BLIND_OUT)